In [1]:
!pip install requests


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install numpy pandas os sys json platform  dash JupyterDash dash_deck

ERROR: Could not find a version that satisfies the requirement os (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for os


In [4]:
!pip install jupyter_dash

  Using cached jupyter_dash-0.4.2-py3-none-any.whl.metadata (3.6 kB)
  Using cached ansi2html-1.9.2-py3-none-any.whl.metadata (3.7 kB)
Using cached jupyter_dash-0.4.2-py3-none-any.whl (23 kB)
Using cached ansi2html-1.9.2-py3-none-any.whl (17 kB)

   ---------------------------------------- 0/2 [ansi2html]
   ---------------------------------------- 2/2 [jupyter_dash]




[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
!pip install dash_deck

  Using cached dash_deck-0.0.1-py3-none-any.whl



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import os, sys, json, platform
from pathlib import Path
import numpy as np
import pandas as pd

from jupyter_dash import JupyterDash
from dash import html, dcc, Output, Input
import dash_bootstrap_components as dbc
import dash_deck

print("Python :", sys.version.split()[0])
print("OS     :", platform.platform())
print("pandas :", pd.__version__)
print("numpy  :", np.__version__)
print("dash-deck available?:", hasattr(dash_deck, "DeckGL"))


Python : 3.12.10
OS     : Windows-11-10.0.26100-SP0
pandas : 2.3.1
numpy  : 2.3.2
dash-deck available?: True


In [8]:
# 🔧 EDIT THESE TWO PATHS
PARQUET_PATH = Path(r"D:\Hitakshi\Dashboard\slm-dashboard\data\eu_labor_force_small.parquet")
GEOJSON_L0_PATH = Path(r"D:\Hitakshi\Dashboard\slm-dashboard\data\NUTS_RG_20M_2024_4326_LEVL_0.geojson")

# Optional basemap (leave blank to render without a basemap)
MAPBOX_TOKEN = os.environ.get("MAPBOX_TOKEN", "")

def norm_code(x):
    if pd.isna(x):
        return None
    return str(x).strip().upper()

def initial_view_state():
    return dict(latitude=54.0, longitude=15.0, zoom=3.3,
                minZoom=2.0, maxZoom=10.5, pitch=0, bearing=0)


In [9]:
df_all = pd.read_parquet(PARQUET_PATH)
req = {"region_level", "region_code", "year", "emp"}
missing = req - set(df_all.columns)
assert not missing, f"Parquet missing required columns: {sorted(missing)}"

df2 = df_all[df_all["region_level"] == 2].copy()
years = sorted(df2["year"].dropna().unique().tolist())
assert years, "No years found for region_level == 2"

print("Rows @ level 2:", len(df2))
print("Available years:", years[:15], "..." if len(years) > 15 else "")
df2.head(3)


Rows @ level 2: 20213
Available years: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014] ...


,year,region_code,region_level,sex_code,sex_level,age_code,age_level,emp
0,2002,AT,2,T,1,Y15-74,1,3792.794
1,2002,AT,2,T,1,Y15-24,2,371.251
2,2002,AT,2,T,1,Y25-34,2,672.493


In [10]:
with open(GEOJSON_L0_PATH, "r", encoding="utf-8") as f:
    geo_l0 = json.load(f)

features = geo_l0.get("features", [])
print("GeoJSON features:", len(features))
if features:
    print("Sample properties keys:", list((features[0].get("properties") or {}).keys())[:20])


GeoJSON features: 39
Sample properties keys: ['NUTS_ID', 'LEVL_CODE', 'CNTR_CODE', 'NAME_LATN', 'NUTS_NAME', 'MOUNT_TYPE', 'URBN_TYPE', 'COAST_TYPE']


In [11]:
parquet_codes = set(df2["region_code"].map(norm_code).dropna().unique())
geo_codes = set()
for feat in features:
    props = feat.get("properties", {}) or {}
    nid = props.get("NUTS_ID") or props.get("nuts_id") or props.get("id") or props.get("ID")
    if nid is not None:
        geo_codes.add(norm_code(nid))

missing_in_geo = sorted(parquet_codes - geo_codes)

print("Unique parquet codes:", len(parquet_codes))
print("Unique geo NUTS_ID  :", len(geo_codes))
print("Parquet NOT in Geo  :", len(missing_in_geo))
if missing_in_geo[:10]:
    print("Examples:", missing_in_geo[:10])

assert len(missing_in_geo) == 0, "Some parquet codes are NOT present in the Level-0 GeoJSON."


Unique parquet codes: 27
Unique geo NUTS_ID  : 39
Parquet NOT in Geo  : 0


In [14]:
!pip install dash_deck


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
from copy import deepcopy
import json

def enrich_static(geo):
    g = deepcopy(geo)
    for feat in g.get("features", []):
        props = feat.setdefault("properties", {})
        nid = norm_code(props.get("NUTS_ID") or props.get("nuts_id") or props.get("id") or props.get("ID"))
        name = props.get("NAME_LATN") or props.get("NAME_2021") or props.get("CNTR_NAME") or ""
        props["nuts_id"] = nid
        props["tooltip"] = f"{name} ({nid})"
    return g

gj_static = enrich_static(geo_l0)

layer_geo = {
    "@@type": "GeoJsonLayer",
    "id": "nuts-level0",
    "data": gj_static,
    "pickable": True,
    "stroked": True,
    "filled": True,
    "opacity": 0.9,
    "getFillColor": [8, 81, 156, 200],    # static fill
    "getLineColor": [80, 80, 80],
    "lineWidthMinPixels": 0.75,
    "autoHighlight": True,
}

smoke_layer = {
    "@@type": "ScatterplotLayer",
    "id": "smoke",
    "data": [
        {"position": [2.3522, 48.8566]},   # Paris
        {"position": [13.4050, 52.5200]},  # Berlin
        {"position": [-0.1276, 51.5074]},  # London
    ],
    "getPosition": "@@=position",
    "getRadius": 40000,
    "radiusMinPixels": 4,
    "getFillColor": [255, 255, 255, 255],
    "pickable": False,
}

deck_spec = {
    "initialViewState": dict(latitude=54.0, longitude=15.0, zoom=3.3,
                             minZoom=2.0, maxZoom=10.5, pitch=0, bearing=0),
    "layers": [layer_geo, smoke_layer],
    "controller": True,
}
if MAPBOX_TOKEN:
    deck_spec["mapStyle"] = "mapbox://styles/mapbox/light-v11"

app_static = JupyterDash(__name__ + "_static", external_stylesheets=[dbc.themes.MINTY])
app_static.layout = dbc.Container(
    [
        dbc.Navbar(dbc.Container([dbc.NavbarBrand("Static polygons + smoke dots")], fluid=True),
                   color="primary", dark=True),
        dbc.Container(
            [
                dash_deck.DeckGL(
                    data=json.dumps(deck_spec),   # 👈 pass the whole Deck spec here
                    tooltip={"text": "{properties.tooltip}"},
                    mapboxKey=MAPBOX_TOKEN if MAPBOX_TOKEN else None,
                    style={"height": "70vh", "width": "100%", "borderRadius": "1rem", "overflow": "hidden"},
                )
            ],
            fluid=True
        ),
    ],
    fluid=True
)

app_static.run(mode="inline", port=8051, debug=True)


In [ ]:
# Choose the year to visualize (latest by default)
sel_year = years[-1]
print("Selected year:", sel_year)

def emp_map_for_year(y):
    sub = df2[df2["year"] == y].c  opy().sort_values(["region_code"])
    return sub.groupby("region_code")["emp"].last().rename(index=norm_code).to_dict()

emp_map = emp_map_for_year(sel_year)
print("Regions with emp data:", sum(pd.notna(list(emp_map.values()))))


Selected year: 2040
Regions with emp data: 27


In [18]:
from copy import deepcopy

def colorize_by_emp(geo, emp_map):
    g = deepcopy(geo)
    vals = np.array([v for v in emp_map.values() if pd.notna(v)], dtype=float)
    if len(vals) == 0:
        return g, [], []

    # 7-class quantiles
    num_classes = 7
    bins = np.quantile(vals, np.linspace(0, 1, num_classes + 1)).astype(float)
    # ensure strictly increasing
    for i in range(1, len(bins)):
        if bins[i] <= bins[i-1]:
            bins[i] = bins[i-1] + 1e-9

    palette = [
        [239, 243, 255],
        [198, 219, 239],
        [158, 202, 225],
        [107, 174, 214],
        [66, 146, 198],
        [33, 113, 181],
        [8, 81, 156],
    ]

    def color_for(x):
        if x is None or pd.isna(x):
            return [220, 220, 220, 180]
        idx = np.searchsorted(bins, float(x), side="right") - 1
        idx = max(0, min(idx, num_classes - 1))
        r, g_, b = palette[idx]
        return [int(r), int(g_), int(b), 200]

    for feat in g.get("features", []):
        props = feat.setdefault("properties", {})
        nid = norm_code(props.get("NUTS_ID") or props.get("nuts_id") or props.get("id") or props.get("ID"))
        name = props.get("NAME_LATN") or props.get("NAME_2021") or props.get("CNTR_NAME") or ""
        ev = emp_map.get(nid, None)
        props["nuts_id"] = nid
        props["emp"] = None if ev is None or pd.isna(ev) else float(ev)
        props["tooltip"] = f"{name} ({nid})\\nemp: {props['emp'] if props['emp'] is not None else 'n/a'}"
        props["fillColor"] = color_for(props["emp"])

    legend_labels = [f"{bins[i]:,.0f} – {bins[i+1]:,.0f}" for i in range(len(bins)-1)]
    return g, palette, legend_labels

gj_color, palette, legend_labels = colorize_by_emp(geo_l0, emp_map)
len(gj_color.get("features", [])), legend_labels[:3]


(39, ['25 – 81', '81 – 150', '150 – 350'])

In [20]:
import json

layer_geo2 = {
    "@@type": "GeoJsonLayer",
    "id": "nuts-level0-colored",
    "data": gj_color,
    "pickable": True,
    "stroked": True,
    "filled": True,
    "opacity": 0.95,
    "getFillColor": "@@=properties.fillColor",
    "getLineColor": [80, 80, 80],
    "lineWidthMinPixels": 0.75,
    "autoHighlight": True,
}

deck_spec2 = {
    "initialViewState": dict(latitude=54.0, longitude=15.0, zoom=3.3,
                             minZoom=2.0, maxZoom=10.5, pitch=0, bearing=0),
    "layers": [layer_geo2],
    "controller": True,
}
if MAPBOX_TOKEN:
    deck_spec2["mapStyle"] = "mapbox://styles/mapbox/light-v11"

legend_children = []
for color, label in zip(palette, legend_labels):
    r, g_, b = color
    legend_children.append(
        html.Div(
            [
                html.Span(style={
                    "display":"inline-block","width":"18px","height":"18px","borderRadius":"4px",
                    "marginRight":"8px","backgroundColor": f"rgb({r},{g_},{b})",
                    "border":"1px solid rgba(0,0,0,0.15)"
                }),
                html.Span(label)
            ],
            className="mb-1"
        )
    )

app_emp = JupyterDash(__name__ + "_emp", external_stylesheets=[dbc.themes.MINTY])
app_emp.layout = dbc.Container(
    [
        dbc.Navbar(dbc.Container([dbc.NavbarBrand(f"Emp choropleth — Year {sel_year}")], fluid=True),
                   color="dark", dark=True),
        dbc.Row(
            [
                dbc.Col(
                    dash_deck.DeckGL(
                        data=json.dumps(deck_spec2),   # 👈 pass the Deck spec here
                        tooltip={"text": "{properties.tooltip}"},
                        mapboxKey=MAPBOX_TOKEN if MAPBOX_TOKEN else None,
                        style={"height": "70vh", "width": "100%", "borderRadius": "1rem", "overflow": "hidden"}
                    ), md=9
                ),
                dbc.Col(
                    dbc.Card([dbc.CardHeader("Legend (emp)"), dbc.CardBody(legend_children)]),
                    md=3
                ),
            ],
            className="g-3"
        ),
    ],
    fluid=True
)
app_emp.run(mode="inline", port=8052, debug=True)
